# 04 Machine Learning Baselines

Project: Predicting Thyroid Dysfunction Using Machine Learning

Author: Namitha Nair

This notebook is part of a cleaned research workflow for the NHANES thyroid-metabolic analysis.


## Purpose

This notebook builds baseline machine learning models to predict elevated TSH using routinely collected demographic and metabolic variables. The goal is not clinical diagnosis, but to evaluate whether these variables contain predictive signal.


In [ ]:
from pathlib import Path

# Project paths. These work if the notebooks are run from the repository root.
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FIGURES_DIR = Path("figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

DATA_PATH = PROCESSED_DIR / "NHANES_Analysis_Dataset.csv"
analysis_df = pd.read_csv(DATA_PATH)

print(analysis_df.shape)
analysis_df.head()


## Define binary prediction task

Low TSH cases are excluded for the first baseline model because the group is small and may represent a different physiological pattern. The binary target is:

- `0` = Normal TSH
- `1` = High TSH


In [ ]:
analysis_binary = analysis_df[analysis_df["TSH_Group"] != "Low"].copy()
analysis_binary["High_TSH"] = (analysis_binary["TSH_Group"] == "High").astype(int)

analysis_binary["High_TSH"].value_counts()


## Prepare features


In [ ]:
features = [
    "Age_Years",
    "Sex",
    "BMI",
    "Fasting_Glucose_mg_dL",
    "HbA1c_Percent",
    "Fasting_Insulin_uU_mL"
]

X = analysis_binary[features].copy()
y = analysis_binary["High_TSH"]

# Convert Sex to numeric if it is stored as text.
if X["Sex"].dtype == "object":
    X["Sex"] = X["Sex"].map({"Male": 1, "Female": 0})

# Median imputation for remaining missing numeric values.
X = X.fillna(X.median(numeric_only=True))

print(X.isnull().sum())


## Train/test split

A stratified split preserves the proportion of High TSH cases in both training and testing sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)
print("
Training distribution")
print(y_train.value_counts())
print("
Testing distribution")
print(y_test.value_counts())


## Model 1: Balanced Logistic Regression


In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred_lr))
print()
print(classification_report(y_test, y_pred_lr))
print("ROC AUC:", roc_auc_score(y_test, y_prob_lr))


In [ ]:
lr_coef = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr.coef_[0]
}).sort_values("Coefficient", ascending=False)

lr_coef


## Model 2: Random Forest


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred_rf))
print()
print(classification_report(y_test, y_pred_rf, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob_rf))


In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

importance


## ROC curve comparison


In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_rf = roc_auc_score(y_test, y_prob_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression AUC = {auc_lr:.3f}")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest AUC = {auc_rf:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "roc_curve_comparison.png", dpi=300)
plt.show()


## Save model comparison summary


In [ ]:
model_results = pd.DataFrame([
    {
        "Model": "Balanced Logistic Regression",
        "ROC_AUC": auc_lr
    },
    {
        "Model": "Random Forest",
        "ROC_AUC": auc_rf
    }
])

model_results.to_csv(PROCESSED_DIR / "model_results.csv", index=False)
importance.to_csv(PROCESSED_DIR / "random_forest_feature_importance.csv", index=False)
lr_coef.to_csv(PROCESSED_DIR / "logistic_regression_coefficients.csv", index=False)

model_results


## Preliminary interpretation

The balanced logistic regression model serves as the baseline. In preliminary testing, it showed modest predictive signal for High TSH. The Random Forest baseline did not clearly improve prediction, suggesting that the current feature set may be more limiting than model complexity. Future work should expand the feature set using clinically relevant NHANES variables such as lipids, blood pressure, inflammatory markers, micronutrients, and medication history.
